#<h1><center>Lab 4 - A4</center></h1>

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import math
import time
import pickle as pk
# Importuri legate de sklearn
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler, Normalizer
from sklearn.model_selection import cross_validate, train_test_split

In [ ]:
# Importuri utile pentru tensorflow
#import tensorflow as tf
#from tensorflow import keras
#from tensorflow.keras import layers

# Importuri utile pentru pytorch
#import torch
#import torch.nn as nn
#import torch.nn.functional as F
#import torch.optim as optim

## Lucru în timpul laboratorului

1.	Completarea de cod în fișierul Jupyter Notebook (A4.ipynb) unde este notat cu TODO:
 1.	Despărțirea seturilor de date în train și test
 1.	Normalizarea datelor
 1.	Completarea de cod pentru arborele de decizie
 1.	Completarea de cod pentru funcția de cross-over a arborelui de decizie și antrenarea acestuia
 1.	Completarea de cod pentru rețeaua neuronală
 1.	Completarea de cod pentru funcția de cross-over a rețelei neuronale și antrenarea acesteia

Punctele se împart în mod egal între problemele de clasificare (Iris) și regresie (Diabetes) pentru fiecare subpunct de la punctul 1.

**Total Punctaj A4-Lab = 60p**

**Deadline Lab 4**


### Clasificare - Iris

### Utils

In [ ]:
scoring = {'accuracy' : make_scorer(accuracy_score),
           'precision' : make_scorer(precision_score, average='weighted', zero_division=0),
           'recall' : make_scorer(recall_score, average='weighted', zero_division=0),
           'f1_score' : make_scorer(f1_score, average='weighted', zero_division=0)}

def get_score_mean(scores):
  for method in scores:
      scores[method] = np.mean(scores[method])

#### Încărcare și separare date

In [ ]:
# Re-rulați începând de la această celulă dacă apar erori de nerezolvat la date
iris = datasets.load_iris()
X = iris.data
y = iris.target

In [ ]:
print(X.shape)
print(y.shape)

(150, 4)
(150,)


In [ ]:
print(X[:5])
print(y[:5])

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
[0 0 0 0 0]


#### Normalizarea datelor și separarea datelor

In [ ]:
# done: Aplică un scaler la alegere pentru datele de intrare
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MaxAbsScaler.html
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
# https://scikit-learn.org/1.6/modules/generated/sklearn.preprocessing.Normalizer.html
scaler = MinMaxScaler() # done: Inițializează scaler
X = scaler.fit_transform(X) # done: Aplica scaler pe datele de intrare

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2) # done: Aplică separarea pe train și test utilizând sklearn
# https://scikit-learn.org/1.6/modules/generated/sklearn.model_selection.train_test_split.html

#### Pregătire model și cross-validation

##### Arbore de decizie

In [ ]:
def get_score_dt(X, y, max_leaf_nodes, scoring):
    """
    Funcție pentru a obține mai multe scoruri folosind un arbore de decizie.
    Astfel putem evalua și setul de date
    """
    my_pipeline = Pipeline(steps=[
                              ('model', DecisionTreeClassifier(
                                  max_leaf_nodes=max_leaf_nodes,
                                  random_state=1, class_weight='balanced'))
                             ]) # done: Completare cod cu datele corecte unde este None
    scores = cross_validate(my_pipeline, X, y,
                            cv=5, scoring=scoring,
                            return_train_score=True, error_score='raise') # done: Completare cod cu datele corecte unde este None
    get_score_mean(scores)
    return scores

In [ ]:
# Verificarea codului de mai sus
results = {}
for i in [5, 10, 50, 100, 500, 1000]:
    results[i] = get_score_dt(x_train, y_train, i, scoring)

best_no_leafs = 5
for result in results:
  if results[result]['test_accuracy'] > results[best_no_leafs]['test_accuracy']:
    best_no_leafs = result
  print(result)
  for k, v in results[result].items():
    print(f"{k}: {float(v)}")
  print("\n")

5
fit_time: 0.0022171974182128907
score_time: 0.006646013259887696
test_accuracy: 0.95
train_accuracy: 0.9854166666666666
test_precision: 0.9553009259259259
train_precision: 0.9861138027272247
test_recall: 0.95
train_recall: 0.9854166666666666
test_f1_score: 0.9497027699968876
train_f1_score: 0.9854030765103626


10
fit_time: 0.0019336700439453124
score_time: 0.006105184555053711
test_accuracy: 0.95
train_accuracy: 1.0
test_precision: 0.955185185185185
train_precision: 1.0
test_recall: 0.95
train_recall: 1.0
test_f1_score: 0.9496273552155905
train_f1_score: 1.0


50
fit_time: 0.0019266605377197266
score_time: 0.006103324890136719
test_accuracy: 0.95
train_accuracy: 1.0
test_precision: 0.955185185185185
train_precision: 1.0
test_recall: 0.95
train_recall: 1.0
test_f1_score: 0.9496273552155905
train_f1_score: 1.0


100
fit_time: 0.0018890380859375
score_time: 0.006063270568847656
test_accuracy: 0.95
train_accuracy: 1.0
test_precision: 0.955185185185185
train_precision: 1.0
test_recall: 0

In [ ]:
my_pipeline = Pipeline(steps=[('model', DecisionTreeClassifier(
                                  max_leaf_nodes=best_no_leafs,
                                  random_state=1, class_weight='balanced'))]) # done: Recreează pipeline folosing cei mai buni parametri
my_pipeline.fit(x_train, y_train) # done: Aplică funcția fit pentru pipeline
trained_decision_tree = my_pipeline['model']
y_pred = trained_decision_tree.predict(x_test)

In [ ]:
# salvează model
with open(f'path.pickle', 'wb') as handle:
    pk.dump(trained_decision_tree, handle, protocol=pk.HIGHEST_PROTOCOL)

In [ ]:
# încarcă model
with open(f'path.pickle', 'rb') as handle:
    trained_decision_tree = pk.load(handle)

##### Rețea neuronală

In [ ]:
def get_score_mlp(X, y, n_hidden_layers, scoring):
    """
    Funcție pentru a obține mai multe scoruri folosind o rețea neuronală.
    Astfel putem evalua și setul de date
    """
    my_pipeline = Pipeline(steps=[
                              ('model', MLPClassifier(solver='lbfgs', alpha=1e-5,
                                                      hidden_layer_sizes=n_hidden_layers,
                                                      random_state=1, max_iter=100))
                             ]) # done: Completare cod cu datele corecte unde este None
    scores = cross_validate(my_pipeline, X, y,
                            cv=5, scoring=scoring,
                            return_train_score=True, error_score='raise')
                            # done: Completare cod cu datele corecte unde este None
    get_score_mean(scores)
    return scores

In [ ]:
# Verificarea codului de mai sus
results = {}
for i in [1, 2, 3, 4, 5, 6]:
    results[i] = get_score_mlp(x_train, y_train, i, scoring)

best_no_hidden_layers = 1
for result in results:
  if results[result]['test_accuracy'] > results[best_no_hidden_layers]['test_accuracy']:
    best_no_hidden_layers = result
  print(result)
  print(results[result])

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS R

1
{'fit_time': np.float64(0.0036826133728027344), 'score_time': np.float64(0.006624174118041992), 'test_accuracy': np.float64(0.3583333333333333), 'train_accuracy': np.float64(0.3583333333333333), 'test_precision': np.float64(0.12881944444444446), 'train_precision': np.float64(0.12842881944444445), 'test_recall': np.float64(0.3583333333333333), 'train_recall': np.float64(0.3583333333333333), 'test_f1_score': np.float64(0.18939393939393936), 'train_f1_score': np.float64(0.18908005480524565)}
2
{'fit_time': np.float64(0.004635667800903321), 'score_time': np.float64(0.00899500846862793), 'test_accuracy': np.float64(0.3583333333333333), 'train_accuracy': np.float64(0.3583333333333333), 'test_precision': np.float64(0.12881944444444446), 'train_precision': np.float64(0.12842881944444445), 'test_recall': np.float64(0.3583333333333333), 'train_recall': np.float64(0.3583333333333333), 'test_f1_score': np.float64(0.18939393939393936), 'train_f1_score': np.float64(0.18908005480524565)}
3
{'fit_ti

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [ ]:

my_pipeline = Pipeline(steps=[('model', MLPClassifier(solver='lbfgs', alpha=1e-5,
                                                      hidden_layer_sizes=tuple([100] * best_no_hidden_layers),
                                                      random_state=1, max_iter=1000))]) # done: Recreează pipeline folosing cei mai buni parametri
my_pipeline.fit(x_train, y_train) # done: Aplică funcția fit pentru pipeline
trained_mlp = my_pipeline['model']
y_pred = trained_mlp.predict(x_test)

In [ ]:
# salvează model
with open(f'path.pickle', 'wb') as handle:
    pk.dump(trained_mlp, handle, protocol=pk.HIGHEST_PROTOCOL)

In [ ]:
# încarcă model
with open(f'path.pickle', 'rb') as handle:
    trained_mlp = pk.load(handle)

### Regresie - Diabetes

#### Utils

In [ ]:
# Funcții pentru a obține acuratețea
scoring = {'mse' : make_scorer(mean_squared_error),
           'r2' : make_scorer(r2_score)}
def get_score_mean(scores):
  for method in scores:
      scores[method] = np.mean(scores[method])

#### Încărcare și separare date

In [ ]:
# Re-rulați începând de la această celulă dacă apar erori de nerezolvat la date
diabetes = datasets.load_diabetes()
X = diabetes.data
y = diabetes.target

In [ ]:
print(X.shape)
print(y.shape)

(442, 10)
(442,)


In [ ]:
print(X[:5])
print(y[:5])

[[ 0.03807591  0.05068012  0.06169621  0.02187239 -0.0442235  -0.03482076
  -0.04340085 -0.00259226  0.01990749 -0.01764613]
 [-0.00188202 -0.04464164 -0.05147406 -0.02632753 -0.00844872 -0.01916334
   0.07441156 -0.03949338 -0.06833155 -0.09220405]
 [ 0.08529891  0.05068012  0.04445121 -0.00567042 -0.04559945 -0.03419447
  -0.03235593 -0.00259226  0.00286131 -0.02593034]
 [-0.08906294 -0.04464164 -0.01159501 -0.03665608  0.01219057  0.02499059
  -0.03603757  0.03430886  0.02268774 -0.00936191]
 [ 0.00538306 -0.04464164 -0.03638469  0.02187239  0.00393485  0.01559614
   0.00814208 -0.00259226 -0.03198764 -0.04664087]]
[151.  75. 141. 206. 135.]


#### Normalizarea datelor și separarea datelor

In [ ]:
# done: Aplică un scaler la alegere pentru datele de intrare
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MaxAbsScaler.html
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
# https://scikit-learn.org/1.6/modules/generated/sklearn.preprocessing.Normalizer.html
scaler = MinMaxScaler() # done: Inițializează scaler
X = scaler.fit_transform(X) # done: Aplica scaler pe datele de intrare

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2) # done: Aplică separarea pe train și test utilizând sklearn
# https://scikit-learn.org/1.6/modules/generated/sklearn.model_selection.train_test_split.html

#### Pregătire model și cross-over

##### Arbore de decizie

In [ ]:
def get_score_dt(X, y, max_leaf_nodes, scoring):
    """
    Funcție pentru a obține mai multe scoruri folosind un arbore de decizie.
    Astfel putem evalua și setul de date
    """
    my_pipeline = Pipeline(steps=[
                              ('model', DecisionTreeClassifier(
                                  max_leaf_nodes=max_leaf_nodes,
                                  random_state=1, class_weight='balanced'))
                             ]) # done: Completare cod cu datele corecte unde este None
    scores = cross_validate(my_pipeline, X, y,
                            cv=5, scoring=scoring,
                            return_train_score=True, error_score='raise') # done: Completare cod cu datele corecte unde este None
    get_score_mean(scores)
    return scores

In [ ]:
# Verificarea codului de mai sus
results = {}
for i in [5, 10, 50, 100, 500, 1000]:
    results[i] = get_score_dt(x_train, y_train, i, scoring)

best_no_leafs = 5
for result in results:
  if results[result]['test_r2'] > results[best_no_leafs]['test_r2']:
    best_no_leafs = result
  print(result)
  print(results[result])

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


5
{'fit_time': np.float64(0.006697416305541992), 'score_time': np.float64(0.0036044597625732424), 'test_mse': np.float64(19477.93102615694), 'train_mse': np.float64(19610.323990677392), 'test_r2': np.float64(-2.1750219408919103), 'train_r2': np.float64(-2.189031614104222)}
10
{'fit_time': np.float64(0.010037803649902343), 'score_time': np.float64(0.004902267456054687), 'test_mse': np.float64(19188.741167002012), 'train_mse': np.float64(18657.583620279176), 'test_r2': np.float64(-2.1274038395709574), 'train_r2': np.float64(-2.033756645992128)}
50
{'fit_time': np.float64(0.04745955467224121), 'score_time': np.float64(0.009976053237915039), 'test_mse': np.float64(11969.73138832998), 'train_mse': np.float64(10146.836754128763), 'test_r2': np.float64(-0.9526404907957977), 'train_r2': np.float64(-0.6500937312053635)}
100
{'fit_time': np.float64(0.02749767303466797), 'score_time': np.float64(0.0040166378021240234), 'test_mse': np.float64(8521.668329979879), 'train_mse': np.float64(5552.161559

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [ ]:
my_pipeline = Pipeline(steps=[('model', DecisionTreeClassifier(max_leaf_nodes=best_no_leafs, random_state=1, class_weight='balanced'))]) # done: Recrează pipeline folosing cei mai buni parametri
my_pipeline.fit(x_train, y_train) # done: Aplică funcția fit pentru pipeline
trained_decision_tree = my_pipeline['model']
y_pred = trained_decision_tree.predict(x_test)

In [ ]:
# salvează model
with open(f'path.pickle', 'wb') as handle:
    pk.dump(trained_decision_tree, handle, protocol=pk.HIGHEST_PROTOCOL)

In [ ]:
# încarcă model
with open(f'path.pickle', 'rb') as handle:
    trained_decision_tree = pk.load(handle)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R2 Score: {r2:.2f}")

Mean Squared Error: 6654.68
R2 Score: -0.34


##### Rețea neuronală

In [ ]:
def get_score_mlp(X, y, n_hidden_layers, scoring):
    """
    Funcție pentru a obține mai multe scoruri folosind o rețea neuronală.
    Astfel putem evalua și setul de date
    """
    my_pipeline = Pipeline(steps=[
                              ('model', MLPClassifier(solver='lbfgs', alpha=1e-5,
                                                      hidden_layer_sizes=n_hidden_layers,
                                                      random_state=1, max_iter=100))
                             ]) # done: Completare cod cu datele corecte unde este None
    scores = cross_validate(my_pipeline, X, y,
                            cv=5, scoring=scoring,
                            return_train_score=True, error_score='raise')
                            # done: Completare cod cu datele corecte unde este None
    get_score_mean(scores)
    return scores

In [ ]:
for i in [1, 2, 3, 4, 5, 6]:
    results[i] = get_score_mlp(x_train, y_train, i, scoring)

best_no_hidden_layers = 1
for result in results:
  if results[result]['test_r2'] > results[best_no_hidden_layers]['test_r2']:
    best_no_hidden_layers = result
  print(result)
  print(results[result])

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the

5
{'fit_time': np.float64(0.34394421577453616), 'score_time': np.float64(0.0035324573516845705), 'test_mse': np.float64(5591.359879275655), 'train_mse': np.float64(5037.257757562088), 'test_r2': np.float64(0.08643246130276024), 'train_r2': np.float64(0.181211416935297)}
10
{'fit_time': np.float64(0.010037803649902343), 'score_time': np.float64(0.004902267456054687), 'test_mse': np.float64(19188.741167002012), 'train_mse': np.float64(18657.583620279176), 'test_r2': np.float64(-2.1274038395709574), 'train_r2': np.float64(-2.033756645992128)}
50
{'fit_time': np.float64(0.04745955467224121), 'score_time': np.float64(0.009976053237915039), 'test_mse': np.float64(11969.73138832998), 'train_mse': np.float64(10146.836754128763), 'test_r2': np.float64(-0.9526404907957977), 'train_r2': np.float64(-0.6500937312053635)}
100
{'fit_time': np.float64(0.02749767303466797), 'score_time': np.float64(0.0040166378021240234), 'test_mse': np.float64(8521.668329979879), 'train_mse': np.float64(5552.161559281

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [ ]:
my_pipeline = Pipeline(steps=[
    ('model', MLPRegressor(
        solver='lbfgs',
        alpha=1e-5,
        hidden_layer_sizes=tuple([100] * best_no_hidden_layers),
        random_state=1,
        max_iter=1000))
])
my_pipeline.fit(x_train, y_train) # done: Aplică funcția fit pentru pipeline
trained_mlp = my_pipeline['model']
y_pred = trained_mlp.predict(x_test)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [ ]:
# salvează model
with open(f'path.pickle', 'wb') as handle:
    pk.dump(trained_mlp, handle, protocol=pk.HIGHEST_PROTOCOL)

In [ ]:
# încarcă model
with open(f'path.pickle', 'rb') as handle:
    trained_mlp = pk.load(handle)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE pe test: {mse:.2f}")
print(f"R2 Score pe test: {r2:.2f}")

MSE pe test: 6654.68
R2 Score pe test: -0.34


## A4 - Temă
1.	Implementarea Random Forest pentru problemele de clasificare și regresie (15p):
2.	La alegere (15p):
  * Aplicarea unei rețele neuronale făcută în pytorch sau tensorflow pe problemele prezentate la lab
  * Aplicarea unei rețele neuronale făcută în pytorch sau tensorflow pe o problemă nouă
    * Posibile aplicări: clasificare de imagini, analiză de text sau generare de date
1.	Scrierea de documentație drept un Jupyter Notebook sau un document word/pdf (30p)
  *	Documentare cod
  *	Explicarea funcțiilor pe pași
  *	Tabel cu rezultate pentru fiecare instanță a problemei prentru minim 5 valori diferite de parametri
        *	Incluzând tabele cu rezultate pentru algoritmi implementați la lab
  *	Vizualizare rezultate între interații
  *	Analiza rezultatelor
        *	Comparare cu soluțiile implementate în timpul laboratorului unde se poate

**Total Punctaj A4-Temă= 60p**

**Deadline Lab 5**


In [4]:
# implementare random forest - clasificare si regresie
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error


# clasificare pt setul de date irisi
iris = datasets.load_iris()
X_iris = iris.data
y_iris = iris.target

# impartim datele: 80% pentru antrenare, 20% pentru testare
x_train_i, x_test_i, y_train_i, y_test_i = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

# testam 5 valori diferite pentru parametrul n_estimators (numarul de arbori)
parametri_arbori = [10, 50, 100, 200, 500]

for n in parametri_arbori:
    # definim modelul
    rf_clf = RandomForestClassifier(n_estimators=n, random_state=42)
    # antrenam modelul
    rf_clf.fit(x_train_i, y_train_i)
    # facem predictii pe setul de test
    predictii = rf_clf.predict(x_test_i)
    # calculam acuratetea
    acuratete = accuracy_score(y_test_i, predictii)
    print(f"Număr arbori: {n:3} | Acuratețe test: {acuratete:.4f}")

print("\n")

# regresie pt setul de date diabet
diabetes = datasets.load_diabetes()
X_diab = diabetes.data
y_diab = diabetes.target

# impartim datele
x_train_d, x_test_d, y_train_d, y_test_d = train_test_split(X_diab, y_diab, test_size=0.2, random_state=42)

for n in parametri_arbori:
    # definim modelul (folosim regressor în loc de classifier)
    rf_reg = RandomForestRegressor(n_estimators=n, random_state=42)
    # antrenam modelul
    rf_reg.fit(x_train_d, y_train_d)
    # facem predictii
    predictii = rf_reg.predict(x_test_d)
    # calculam eroarea patratica medie (MSE), cu cat mai mica, cu atat mai bine
    mse = mean_squared_error(y_test_d, predictii)
    print(f"Număr arbori: {n:3} | MSE test: {mse:.4f}")

Număr arbori:  10 | Acuratețe test: 1.0000
Număr arbori:  50 | Acuratețe test: 1.0000
Număr arbori: 100 | Acuratețe test: 1.0000
Număr arbori: 200 | Acuratețe test: 1.0000
Număr arbori: 500 | Acuratețe test: 1.0000


Număr arbori:  10 | MSE test: 3135.2893
Număr arbori:  50 | MSE test: 3044.1991
Număr arbori: 100 | MSE test: 2952.0106
Număr arbori: 200 | MSE test: 2966.0242
Număr arbori: 500 | MSE test: 2993.2969


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

# retea neuronala - clasificare irisi

# normalizam datele
scaler_iris = StandardScaler()
X_iris_scaled = scaler_iris.fit_transform(X_iris)

# impartim datele normalizate
x_train_i, x_test_i, y_train_i, y_test_i = train_test_split(X_iris_scaled, y_iris, test_size=0.2, random_state=42)

# convertim datele in tensori pytorch
X_train_t = torch.tensor(x_train_i, dtype=torch.float32)
y_train_t = torch.tensor(y_train_i, dtype=torch.long) # .long e necesar pt clasificare
X_test_t = torch.tensor(x_test_i, dtype=torch.float32)
y_test_t = torch.tensor(y_test_i, dtype=torch.long)

# definim structura retelei neuronale
class IrisNet(nn.Module):
    def __init__(self, hidden_neurons):
        super(IrisNet, self).__init__()
        # 4 inputs (trasaturile irisilor), N neuroni ascunsi, 3 outputs (clasele)
        self.fc1 = nn.Linear(4, hidden_neurons)
        self.fc2 = nn.Linear(hidden_neurons, 3)
        self.relu = nn.ReLU() # fct de activare

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# testam 5 dimensiuni diferite pentru stratul ascuns
optiuni_neuroni_ascunsi = [4, 8, 16, 32, 64]

for hidden in optiuni_neuroni_ascunsi:
    model = IrisNet(hidden_neurons=hidden)
    # fct de pierdere (loss) și optimizer-ul (cel care actualizeaza ponderile)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # bucla de antrenare (epoci)
    epoci = 100
    for epoca in range(epoci):
        optimizer.zero_grad() # resetam gradientii
        output = model(X_train_t) # trecem datele prin retea
        loss = criterion(output, y_train_t) # calculam eroarea
        loss.backward() # propagam eroarea inapoi
        optimizer.step() # actualizam reteaua

    # evaluare pe setul de test
    with torch.no_grad(): # nu mai calculam gradienti la testare
        test_output = model(X_test_t)
        predictii = torch.argmax(test_output, dim=1) # alegem clasa cu probabilitatea maxima
        acuratete = accuracy_score(y_test_t.numpy(), predictii.numpy())
        print(f"Neuroni ascunsi: {hidden:2} | Acuratete test: {acuratete:.4f}")

print("\n")

# retea neuronala - regresie diabet
scaler_diab = StandardScaler()
X_diab_scaled = scaler_diab.fit_transform(X_diab)

x_train_d, x_test_d, y_train_d, y_test_d = train_test_split(X_diab_scaled, y_diab, test_size=0.2, random_state=42)

# convertim datele in tensori
X_train_d_t = torch.tensor(x_train_d, dtype=torch.float32)
y_train_d_t = torch.tensor(y_train_d, dtype=torch.float32).view(-1, 1) # formatam outputul drept coloana
X_test_d_t = torch.tensor(x_test_d, dtype=torch.float32)
y_test_d_t = torch.tensor(y_test_d, dtype=torch.float32).view(-1, 1)

class DiabetesNet(nn.Module):
    def __init__(self, hidden_neurons):
        super(DiabetesNet, self).__init__()
        # 10 inputs (trasaturi), N neuroni ascunsi, 1 output (valoarea de prezis)
        self.fc1 = nn.Linear(10, hidden_neurons)
        self.fc2 = nn.Linear(hidden_neurons, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

for hidden in optiuni_neuroni_ascunsi:
    model = DiabetesNet(hidden_neurons=hidden)
    # folosim MSELoss pentru regresie
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.05)

    epoci = 150
    for epoca in range(epoci):
        optimizer.zero_grad()
        output = model(X_train_d_t)
        loss = criterion(output, y_train_d_t)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        predictii = model(X_test_d_t)
        mse = mean_squared_error(y_test_d_t.numpy(), predictii.numpy())
        print(f"Neuroni ascunsi: {hidden:2} | MSE test: {mse:.4f}")

Neuroni ascunsi:  4 | Acuratete test: 0.9667
Neuroni ascunsi:  8 | Acuratete test: 1.0000
Neuroni ascunsi: 16 | Acuratete test: 1.0000
Neuroni ascunsi: 32 | Acuratete test: 1.0000
Neuroni ascunsi: 64 | Acuratete test: 1.0000


Neuroni ascunsi:  4 | MSE test: 2909.2214
Neuroni ascunsi:  8 | MSE test: 3047.1707
Neuroni ascunsi: 16 | MSE test: 2845.9158
Neuroni ascunsi: 32 | MSE test: 2819.9358
Neuroni ascunsi: 64 | MSE test: 2743.7280


Random Forest a fost mai ușor de implementat și nu a necesitat normalizarea datelor (arborii de decizie sunt imuni la asta), în timp ce rețelele neuronale au necesitat normalizarea datelor de intrare si in general o implementare mai complexa.